# Bates calibration

Reproducible calibration of the eight-parameter Bates model on the local-only
current LSE GLD Chebyshev sample. Pricing is maturity-batched with COS; scalar Fourier
inversion remains the regression reference.


In [ ]:
import json
from pathlib import Path

from Bates import Bates
from calibration_workflow import (
    load_calibration_surface, option_diagnostics, plot_residuals,
    plot_smiles, price_surface,
)

DATA = Path("Data/lse_local")
SEED = 20260811
DIVIDEND_YIELD = 0.0
surface, spot = load_calibration_surface(DATA, DIVIDEND_YIELD)
surface.head()


In [ ]:
report = Bates.calibrate_bates(
    surface,
    spot,
    q=DIVIDEND_YIELD,
    seed=SEED,
    pricing="cos",
    return_report=True,
)
report.as_dict()


In [ ]:
parameters = report.x
model_prices = price_surface(
    surface,
    lambda strikes, maturity, rate: Bates.bates_prices_cos(
        spot, strikes, maturity, *parameters, rate, DIVIDEND_YIELD
    ),
)
diagnostics = option_diagnostics(
    surface, spot, model_prices, DIVIDEND_YIELD
)
diagnostics.to_csv(DATA / "bates_option_diagnostics.csv", index=False)
(DATA / "bates_calibration_report.json").write_text(
    json.dumps(report.as_dict(), indent=2), encoding="utf-8"
)
diagnostics.head()


In [ ]:
plot_smiles(diagnostics, "Bates", DATA / "bates_volatility_smile.png")
plot_residuals(diagnostics, "Bates", DATA / "bates_residual_heatmap.png")
